In [1]:
!curl https://rclone.org/install.sh | sudo bash


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  4734  100  4734    0     0  10173      0 --:--:-- --:--:-- --:--:-- 10158






































































In [2]:
!mkdir -p /root/.config/rclone
!cp /kaggle/input/datasets/ngc2222/sdsadasdasd/rclone.conf /root/.config/rclone/rclone.conf
!rclone lsd rclone:

           0 2025-04-09 11:22:08        -1 .ipynb_checkpoints
           0 2025-02-16 15:08:56        -1 Colab Notebooks
           0 2025-04-03 06:18:20        -1 Google AI Studio
           0 2016-07-26 15:31:41        -1 LabanKey
           0 2016-07-26 15:31:41        -1 LabanKey
           0 2025-05-07 03:36:28        -1 ML
           0 2025-04-10 07:58:07        -1 WorkingPlant
           0 2025-09-03 01:56:09        -1 faster-rcnn-docker
           0 2025-05-16 16:19:12        -1 leaf_small
           0 2025-02-16 15:08:44        -1 rice_project
           0 2026-02-01 09:22:34        -1 sign_language_models
           0 2025-04-23 09:00:15        -1 swe201c pe
           0 2026-04-09 02:33:59        -1 vinbigdata-CLAHE-png
           0 2026-04-12 06:37:31        -1 vinbigdata-CLAHE_padding-png
           0 2026-04-09 08:22:27        -1 vinbigdata-HistogramEq-png
           0 2026-04-23 08:09:53        -1 vinbigdata-Histogram_EQ_512-png
           0 2026-04-08 07:06:08        -1

In [3]:
import os
import json
import time
import random
import shutil
import subprocess
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import cv2
import numpy as np
import pandas as pd
import pydicom

# =========================
# PATH CONFIG
# =========================
DATA_ROOT = Path("/kaggle/input/competitions/vinbigdata-chest-xray-abnormalities-detection")
TRAIN_ANN_CSV = Path("/kaggle/input/datasets/benxelua/correct-label/annotations/annotations_train.csv")
VAL_ANN_CSV = Path("/kaggle/input/datasets/benxelua/correct-label/annotations/annotations_test.csv")
TRAIN_LABELS_CSV = Path("/kaggle/input/datasets/benxelua/correct-label/annotations/image_labels_train.csv")
VAL_LABELS_CSV = Path("/kaggle/input/datasets/benxelua/correct-label/annotations/image_labels_test.csv")

OUT_ROOT = Path("out_vindr_mmdet_percentile_512_2")
IMG_DIR = OUT_ROOT / "images"
ANN_DIR = OUT_ROOT / "annotations"

IMG_TRAIN_DIR = IMG_DIR / "train"
IMG_VAL_DIR = IMG_DIR / "val"

# Thư mục tạm để chứa file ZIP trước khi đẩy lên rclone
ZIP_DIR = Path("temp_zips")

for d in [IMG_TRAIN_DIR, IMG_VAL_DIR, ANN_DIR, ZIP_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# =========================
# SCRIPT CONFIG
# =========================
SEED = 42
IMG_SIZE = 512
SAVE_AS_JPG = False
JPG_QUALITY = 95
STACK_CHANNELS_3 = True

LOW_PERCENTILE = 0.5
HIGH_PERCENTILE = 99.5

# =========================
# RCLONE CONFIG
# =========================
BATCH_SIZE = 3000
# Thư mục đích trên Google Drive
RCLONE_DEST = "rclone:vinbigdata-percentile-png-512-2" 

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)

seed_everything(SEED)


# Hàm upload thư mục chứa file ZIP bằng rclone
def upload_folder_with_rclone(folder_path, remote_path, max_retries=3, retry_delay=10):
    print(f"📦 Đang đẩy {folder_path} lên {remote_path}...")
    
    # Lệnh copy: Vì bây giờ chỉ upload 1 file ZIP to, nên không cần --tpslimit nữa
    cmd = [
        "rclone", "copy", 
        str(folder_path), 
        remote_path, 
        "--transfers", "4",         # Copy tối đa 4 file ZIP cùng lúc (nếu có)
        "--checkers", "4",
        "--retries", "3",           
        "--stats", "10s"            
    ]
    
    for attempt in range(1, max_retries + 1):
        try:
            subprocess.run(cmd, check=True)
            print(f"✅ Đã upload thành công lên {remote_path} (Lần thử {attempt})")
            return
        except subprocess.CalledProcessError as e:
            print(f"⚠️ [Lần thử {attempt}/{max_retries}] Lỗi rclone: {e}")
            if attempt < max_retries:
                time.sleep(retry_delay)
                retry_delay *= 2 
            else:
                raise Exception("DỪNG CHƯƠNG TRÌNH: Upload thất bại sau nhiều lần thử.")


def list_dicom_files(folder: Path):
    files = []
    for p in sorted(folder.rglob("*")):
        if p.is_file() and p.suffix.lower() in {".dicom", ".dcm"}:
            files.append(p)
    return files


def read_dicom_percentile_clipping(path: Path, low=0.5, high=99.5):
    ds = pydicom.dcmread(str(path))
    img = ds.pixel_array.astype(np.float32)

    # BƯỚC 1: Rescale trước để đưa về đơn vị chuẩn (HU hoặc pixel thực)
    slope = float(getattr(ds, "RescaleSlope", 1.0))
    intercept = float(getattr(ds, "RescaleIntercept", 0.0))
    img = img * slope + intercept

    # BƯỚC 2: Xử lý Photometric (Đưa về Bone = Trắng, Lung = Đen)
    # Phải làm sau khi Rescale vì Max(img) lúc này mới chuẩn
    if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":
        img = np.max(img) - img

    # BƯỚC 3: Tính Percentile trên dải giá trị đã chuẩn hóa sơ bộ
    lo = np.percentile(img, low)
    hi = np.percentile(img, high)
    
    if hi <= lo:
        hi = lo + 1.0
    
    # BƯỚC 4: Clipping và đưa về 8-bit
    img = np.clip(img, lo, hi)
    img = (img - lo) / (hi - lo) # Đưa về [0, 1]
    img = (img * 255.0).astype(np.uint8)
    
    return img


def resize_image_keep_shape(img, size=1024):
    h, w = img.shape[:2]
    resized = cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA)
    return resized, w, h


def sanitize_pascal_voc_boxes(boxes, width, height):
    sanitized = []
    for (x1, y1, x2, y2, class_id) in boxes:
        x1 = float(np.clip(x1, 0, width))
        y1 = float(np.clip(y1, 0, height))
        x2 = float(np.clip(x2, 0, width))
        y2 = float(np.clip(y2, 0, height))
        if x2 <= x1 or y2 <= y1: continue
        sanitized.append((x1, y1, x2, y2, class_id))
    return sanitized


def scale_boxes_to_imgsize(boxes, orig_w, orig_h, img_size):
    scaled = []
    if len(boxes) == 0: return scaled
    sx = img_size / orig_w
    sy = img_size / orig_h
    for (x1, y1, x2, y2, class_id) in boxes:
        x1 = float(np.clip(x1 * sx, 0, img_size))
        y1 = float(np.clip(y1 * sy, 0, img_size))
        x2 = float(np.clip(x2 * sx, 0, img_size))
        y2 = float(np.clip(y2 * sy, 0, img_size))
        if x2 <= x1 or y2 <= y1: continue
        scaled.append((x1, y1, x2, y2, class_id))
    return scaled


def save_image(img, out_path: Path):
    if STACK_CHANNELS_3 and img.ndim == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    if SAVE_AS_JPG:
        cv2.imwrite(str(out_path), img, [cv2.IMWRITE_JPEG_QUALITY, JPG_QUALITY])
    else:
        cv2.imwrite(str(out_path), img, [cv2.IMWRITE_PNG_COMPRESSION, 0])


def process_one_dicom(dicom_path: Path, split: str, boxes):
    image_id = dicom_path.stem
    try:
        img_raw = read_dicom_percentile_clipping(dicom_path, low=LOW_PERCENTILE, high=HIGH_PERCENTILE)
    except Exception as e:
        print(f"Error reading {dicom_path}: {e}")
        return []

    samples = []
    img_resized, orig_w, orig_h = resize_image_keep_shape(img_raw, size=IMG_SIZE)
    boxes = sanitize_pascal_voc_boxes(boxes, width=orig_w, height=orig_h)
    ext = ".jpg" if SAVE_AS_JPG else ".png"
    out_name = f"{image_id}{ext}"
    out_dir = IMG_TRAIN_DIR if split == "train" else IMG_VAL_DIR
    out_path = out_dir / out_name
    save_image(img_resized, out_path)
    scaled_boxes = scale_boxes_to_imgsize(boxes, orig_w, orig_h, IMG_SIZE)
    samples.append({
        "split": split,
        "file_name": f"images/{split}/{out_name}",
        "width": IMG_SIZE,
        "height": IMG_SIZE,
        "boxes": scaled_boxes,
    })
    return samples


def build_coco(samples, class_names):
    images = []
    annotations = []
    categories = [{"id": i + 1, "name": n} for i, n in enumerate(class_names)]
    ann_id = 1
    for img_id, s in enumerate(samples, start=1):
        images.append({
            "id": img_id,
            "file_name": s["file_name"],
            "width": s["width"],
            "height": s["height"],
        })
        for (x1, y1, x2, y2, class_id) in s["boxes"]:
            w, h = float(max(0.0, x2 - x1)), float(max(0.0, y2 - y1))
            if w <= 0 or h <= 0: continue
            annotations.append({
                "id": ann_id, "image_id": img_id, "category_id": int(class_id) + 1,
                "bbox": [float(x1), float(y1), w, h], "area": w * h, "iscrowd": 0,
            })
            ann_id += 1
    return {"images": images, "annotations": annotations, "categories": categories}

def chunker(seq, size):
    return (seq[pos:pos + size] for pos in range(0, len(seq), size))

def main():
    paths = [TRAIN_ANN_CSV, VAL_ANN_CSV, TRAIN_LABELS_CSV, VAL_LABELS_CSV]
    for p in paths:
        if not p.exists():
            print(f"File not found: {p}")
            return

    df_train_ann = pd.read_csv(TRAIN_ANN_CSV)
    df_val_ann = pd.read_csv(VAL_ANN_CSV)
    df_train_labels = pd.read_csv(TRAIN_LABELS_CSV)
    df_val_labels = pd.read_csv(VAL_LABELS_CSV)

    class_names = sorted(list(set(df_train_ann["class_name"].unique()) | set(df_val_ann["class_name"].unique())))
    if "No finding" in class_names: class_names.remove("No finding")
    class2id = {name: i for i, name in enumerate(class_names)}

    dicom_files_all = list_dicom_files(DATA_ROOT)
    all_file_ids = {p.stem for p in dicom_files_all}
    print(f"Total DICOMs found in folder: {len(all_file_ids)}")

    train_ids_all = set(df_train_labels["image_id"].unique()) & all_file_ids
    val_ids_all = set(df_val_labels["image_id"].unique()) & all_file_ids
    
    remaining_ids = all_file_ids - train_ids_all - val_ids_all
    if remaining_ids:
        train_ids_all.update(remaining_ids)

    id_to_split = {}
    for x in train_ids_all: id_to_split[x] = "train"
    for x in val_ids_all: id_to_split[x] = "val"

    def build_ann_map(df):
        ann_map = {}
        df_objs = df[df["class_name"].fillna("").str.lower() != "no finding"]
        for r in df_objs.itertuples(index=False):
            if r.image_id in all_file_ids:
                cid = class2id.get(r.class_name)
                if cid is not None:
                    ann_map.setdefault(r.image_id, []).append((r.x_min, r.y_min, r.x_max, r.y_max, cid))
        return ann_map

    train_ann_map = build_ann_map(df_train_ann)
    val_ann_map = build_ann_map(df_val_ann)

    to_process = [p for p in dicom_files_all if p.stem in id_to_split]
    total_files = len(to_process)
    print(f"Images to process: {total_files} (Train: {len(train_ids_all)}, Val: {len(val_ids_all)})")

    max_workers = min(12, os.cpu_count() or 4)
    print(f"Using max_workers = {max_workers}")

    train_samples, val_samples = [], []
    done_all = 0
    start_all = time.perf_counter()

    # ====================================================================
    # LẶP QUA TỪNG BATCH: XỬ LÝ -> ZIP -> UPLOAD -> DỌN DẸP
    # ====================================================================
    for batch_idx, batch_files in enumerate(chunker(to_process, BATCH_SIZE)):
        print(f"\n--- BẮT ĐẦU XỬ LÝ MẺ {batch_idx + 1} ({len(batch_files)} ảnh) ---")
        
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = []
            for p in batch_files:
                image_id = p.stem
                split = id_to_split[image_id]
                boxes = train_ann_map.get(image_id, []) if split == "train" else val_ann_map.get(image_id, [])
                futures.append(executor.submit(process_one_dicom, p, split, boxes))

            for future in as_completed(futures):
                try:
                    samples = future.result()
                    for s in samples:
                        if s["split"] == "train": train_samples.append(s)
                        else: val_samples.append(s)
                except Exception as e:
                    print("Failed:", e)
                
                done_all += 1
                if done_all % 500 == 0:
                    print(f"Tiến độ tổng: {done_all}/{total_files}...")

        # 1. Nén toàn bộ folder OUT_ROOT/images thành file batch_X.zip lưu vào ZIP_DIR
        zip_base_path = str(ZIP_DIR / f"images_batch_{batch_idx + 1}")
        print(f"🗜️ Đang nén mẻ ảnh thành file ZIP...")
        shutil.make_archive(zip_base_path, 'zip', str(OUT_ROOT))
        
        # 2. Upload thư mục ZIP_DIR (chỉ chứa 1 file zip vừa tạo) lên Google Drive
        upload_folder_with_rclone(folder_path=ZIP_DIR, remote_path=RCLONE_DEST)

        # 3. Dọn dẹp sạch sẽ để nhường chỗ cho mẻ sau
        print("🧹 Đang dọn dẹp file ZIP và ảnh local để giải phóng RAM/Disk...")
        
        # Xóa file ZIP vừa up
        for f in ZIP_DIR.glob("*.zip"):
            f.unlink()
            
        # Xóa ảnh raw ở mẻ này
        for d in [IMG_TRAIN_DIR, IMG_VAL_DIR]:
            for file_path in d.glob("*"):
                if file_path.is_file():
                    file_path.unlink() 
                    
        elapsed = time.perf_counter() - start_all
        print(f"✅ Hoàn thành mẻ {batch_idx + 1}. Tốc độ trung bình: {done_all/elapsed:.2f} img/s")


    # ====================================================================
    # BƯỚC CUỐI: TẠO FILE JSON COCO VÀ UPLOAD
    # ====================================================================
    print("\n📝 Đã xử lý xong toàn bộ ảnh! Bắt đầu tạo file Annotations (JSON)...")
    with open(ANN_DIR / "instances_train.json", "w") as f:
        json.dump(build_coco(train_samples, class_names), f)
    with open(ANN_DIR / "instances_val.json", "w") as f:
        json.dump(build_coco(val_samples, class_names), f)

    # Đẩy nguyên cái folder annotations/ lên Google Drive
    print("🚀 Tải lên folder Annotations lên Google Drive (Lần cuối)...")
    upload_folder_with_rclone(folder_path=ANN_DIR, remote_path=f"{RCLONE_DEST}/annotations")

    print(f"🎉 HOÀN THÀNH TOÀN BỘ QUY TRÌNH!")

if __name__ == "__main__":
    main()

Total DICOMs found in folder: 18000
Images to process: 18000 (Train: 15000, Val: 3000)
Using max_workers = 4

--- BẮT ĐẦU XỬ LÝ MẺ 1 (3000 ảnh) ---
Tiến độ tổng: 500/18000...
Tiến độ tổng: 1000/18000...
Tiến độ tổng: 1500/18000...
Tiến độ tổng: 2000/18000...
Tiến độ tổng: 2500/18000...
Tiến độ tổng: 3000/18000...
🗜️ Đang nén mẻ ảnh thành file ZIP...
📦 Đang đẩy temp_zips lên rclone:vinbigdata-percentile-png-512-2...
✅ Đã upload thành công lên rclone:vinbigdata-percentile-png-512-2 (Lần thử 1)
🧹 Đang dọn dẹp file ZIP và ảnh local để giải phóng RAM/Disk...
✅ Hoàn thành mẻ 1. Tốc độ trung bình: 1.21 img/s

--- BẮT ĐẦU XỬ LÝ MẺ 2 (3000 ảnh) ---
Tiến độ tổng: 3500/18000...
Tiến độ tổng: 4000/18000...
Tiến độ tổng: 4500/18000...
Tiến độ tổng: 5000/18000...
Tiến độ tổng: 5500/18000...
Tiến độ tổng: 6000/18000...
🗜️ Đang nén mẻ ảnh thành file ZIP...
📦 Đang đẩy temp_zips lên rclone:vinbigdata-percentile-png-512-2...
✅ Đã upload thành công lên rclone:vinbigdata-percentile-png-512-2 (Lần thử 1)
🧹 